In [ ]:
# Importo le librerie

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt 
import seaborn as sns
sns.set_theme()

df = pd.read_csv('raw/telco_customer_churn.csv')

In [ ]:
# Valuto informazioni generiche del dataset
df.info()
df.isnull().sum()
df.head()

In [ ]:
# Elimino customerID in quanto non significativo per la mia applicazione
df = df.drop(['customerID'], axis=1)

In [ ]:
# Indago sul perchè TotalCharges è di tipo stringa e non numerico, 
# cercando al suo interno gli elementi di tipo non-numerico

total_charges = pd.to_numeric(df['TotalCharges'], errors='coerce')
nan_indexes = []
for index, charge in enumerate(total_charges):
    if np.isnan(charge):
        nan_indexes.append(index)

df.iloc[nan_indexes]

In [ ]:
# Questi sono tutti clienti da meno di un mese (Tenure == 0) che quindi
# non hanno ancora pagato. 
# Verifico che non vi siano altri utenti con Tenure != 0 o TotalCharges == 0
# , e se non ne trovo impongo TotalCharges = 0
# print(len(df[df['TotalCharges'] == 0]))
if len(df[df['TotalCharges'] == 0]) == 0:
    print("Nessun record con TotalCharges == 0")

df[df['tenure'] == 0]


In [ ]:
# I record individuati sopra sono gli stessi con TotalCharges vuoto, quindi procedo a 
# modificare il dataset
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
df['TotalCharges'] = df['TotalCharges'].fillna(0)

# Esporto il dataset "pulito"
df.to_csv("clean/telco_customer_churn_clean.csv", index = False)

In [ ]:
# Esporto i dataset di training e di validation

from sklearn.model_selection import train_test_split

X = df.drop(columns=["Churn"])
y = df['Churn']
print(X, y)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

X_train["Churn"] = y_train
X_test["Churn"] = y_test

X_train.to_csv("clean/training_set.csv", index = False)
X_test.to_csv("clean/test_set.csv", index = False)
